# 03 - XGBoost Modeling

This notebook trains an XGBoost classifier on the same processed dataset and stratified holdout split used by the logistic-regression baseline. Its purpose is to model nonlinearities and feature interactions that a linear baseline may miss.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from churn_ml.models.evaluate_model import classification_metrics

RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
# CHURN_THRESHOLD = 0.5  # Set a value from 0 to 1 to override the training churn-rate threshold.
def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

DATA_PATH = find_project_file(Path("data/processed/prediction_df_xgboost.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))

model_df = pd.read_csv(DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert model_df.columns[-1] == TARGET_COLUMN, "The target must be the final column."
X = model_df.drop(columns=TARGET_COLUMN)
y = model_df[TARGET_COLUMN]
if len(value_df) != len(model_df) or not value_df[TARGET_COLUMN].equals(y):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")
customer_ltv = value_df.set_index("CustomerID")["CLTV"].rename("predicted_ltv_if_retained")
customer_ids = value_df["CustomerID"]
X_train, X_test, y_train, y_test, ltv_train, ltv_test, customer_id_train, customer_id_test = train_test_split(
    X, y, customer_ltv, customer_ids, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Positive-class weight: {scale_pos_weight:.2f}")
print(f"Prediction threshold: {decision_threshold:.1%}")

Training rows: 5,634; test rows: 1,409
Positive-class weight: 2.77
Prediction threshold: 26.5%


## Train and evaluate

The parameters below are deliberately conservative starting values, not a tuned final model. XGBoost natively handles the missing values in `Total Charges`; the missingness indicator remains available as an explicit feature.

In [2]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
xgb_model.fit(X_train, y_train)

y_proba = xgb_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= decision_threshold).astype(int)
xgb_metrics = pd.Series(classification_metrics(y_test, y_pred, y_proba), name="xgboost")
display(xgb_metrics.to_frame())

confusion = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion, x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues", text=confusion, texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"XGBoost Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

,xgboost
accuracy,0.688432
precision,0.455902
recall,0.898396
f1,0.604860
pr_auc,0.666765
roc_auc,0.847525


## Feature importance

Gain-based importance summarizes how much each feature improves tree splits. It is a model-specific association measure, not a causal explanation.

In [3]:
feature_importance = (
    pd.DataFrame({"feature": X_train.columns, "importance": xgb_model.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
feature_importance.head(15)

,feature,importance
0,Contract_Month-to-month,0.303364
1,Online Security_No,0.082840
2,Internet Service_Fiber optic,0.076467
3,Dependents,0.058257
4,Tech Support_No,0.043698
5,Contract_Two year,0.035857
6,Streaming Movies_Yes,0.034663
7,Online Security_No internet service,0.033905
8,Payment Method_Electronic check,0.027879
9,Internet Service_DSL,0.024730


# Expected Value of Retention Targeting

This evaluation retrieves the raw dataset's `CLTV` value for every holdout-test customer and treats it as that customer's **predicted lifetime value if retained**. `CLTV` is deliberately not a churn-model feature; it is used only after prediction to prioritize outreach.

The expected net value for each customer is calculated as:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

Assumptions: outreach costs **$20** for every targeted customer. The offer costs **$500** only when it is accepted; this scenario assumes a **40% offer-acceptance rate among targeted customers**, including customers who would have stayed without outreach. Its expected cost is therefore $200 per target. The 10% retention uplift is a separate scenario assumption: it represents the share of would-be churners saved by the intervention. Neither assumption is estimated by the churn model, so replace them when campaign data becomes available. The top 100 are selected by expected net value—not merely by churn probability—so high-value customers are prioritized.

In [4]:
OUTREACH_COST = 20
OFFER_COST = 500
RETENTION_UPLIFT = 0.10
OFFER_ACCEPTANCE_RATE = 0.40
TARGET_COUNT = 100

targeting_candidates = pd.DataFrame({
    "CustomerID": customer_id_test.to_numpy(),
    "predicted_churn_probability": y_proba,
    "predicted_ltv_if_retained": ltv_test.to_numpy(),
})
targeting_candidates["expected_value_before_cost"] = (
    targeting_candidates["predicted_churn_probability"]
    * RETENTION_UPLIFT
    * targeting_candidates["predicted_ltv_if_retained"]
)
targeting_candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
targeting_candidates["campaign_cost"] = OUTREACH_COST + targeting_candidates["expected_offer_cost"]
targeting_candidates["expected_net_value"] = (
    targeting_candidates["expected_value_before_cost"]
    - targeting_candidates["campaign_cost"]
)

top_100_targets = (
    targeting_candidates
    .sort_values("expected_net_value", ascending=False)
    .head(TARGET_COUNT)
    .reset_index(drop=True)
)

targeting_summary = pd.DataFrame({
    "customers_targeted": [len(top_100_targets)],
    "expected_value_before_cost": [top_100_targets["expected_value_before_cost"].sum()],
    "outreach_cost": [OUTREACH_COST * len(top_100_targets)],
    "expected_offer_cost": [top_100_targets["expected_offer_cost"].sum()],
    "campaign_cost": [top_100_targets["campaign_cost"].sum()],
    "expected_net_value": [top_100_targets["expected_net_value"].sum()],
})
display(targeting_summary.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))
top_100_targets.style.format({
    "predicted_churn_probability": "{:.1%}",
    "predicted_ltv_if_retained": "${:,.0f}",
    "expected_value_before_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
})

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
0,100,"$45,390.09","$2,000.00","$20,000.00","$22,000.00","$23,390.09"


,CustomerID,predicted_churn_probability,predicted_ltv_if_retained,expected_value_before_cost,expected_offer_cost,campaign_cost,expected_net_value
0,0295-PPHDO,98.2%,"$5,962",$585.67,$200.00,$220.00,$365.67
1,5178-LMXOP,98.2%,"$5,795",$568.85,$200.00,$220.00,$348.85
2,3716-BDVDB,95.7%,"$5,795",$554.64,$200.00,$220.00,$334.64
3,1628-BIZYP,94.3%,"$5,754",$542.70,$200.00,$220.00,$322.70
4,2865-TCHJW,93.1%,"$5,808",$540.80,$200.00,$220.00,$320.80
5,7180-PISOG,96.2%,"$5,554",$534.09,$200.00,$220.00,$314.09
6,6651-AZVTJ,87.6%,"$6,088",$533.44,$200.00,$220.00,$313.44
7,8361-LTMKD,90.7%,"$5,839",$529.72,$200.00,$220.00,$309.72
8,1320-HTRDR,89.0%,"$5,948",$529.39,$200.00,$220.00,$309.39
9,6910-HADCM,94.0%,"$5,613",$527.85,$200.00,$220.00,$307.85
